# FMP Evidence-Based Daily-Adjusted Fundamentals

This notebook tests whether daily-adjusted valuation features built from FMP historical data add signal versus static/as-of quarterly fundamentals.

Rules for this analysis:

- Use only data already stored in Quant Warehouse.
- Use historical data only: daily prices, daily historical market cap, quarterly income/balance/cash/ratios/metrics.
- Do not use snapshot endpoints such as share statistics.
- Do not call the FMP API directly.
- Do not use `historical_enterprise_value`; daily enterprise value is reconstructed from daily market cap plus last-known debt minus last-known cash.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from quant_warehouse import Warehouse

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

PROVIDER = "fmp"
MAG7 = ("AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "TSLA")
# Start small: three symbols and a few recent quarters. Scale only after this passes.
ANALYSIS_SYMBOLS = ("AAPL", "MSFT", "NVDA")
START_DATE = "2024-01-01"
END_DATE = None
FILING_LAG_DAYS = 45
HORIZONS = (20, 60, 120)
MIN_OBS = 120

FULL_RUN_BASELINE_SYMBOLS = MAG7
FULL_RUN_BASELINE_START_DATE = "2018-01-01"

wh = Warehouse()
run_timings: dict[str, float] = {}


## Feature Families

The notebook compares four groups:

1. **Daily market-cap adjusted valuation**: last-known fundamentals divided by daily historical market cap, or the inverse.
2. **Daily reconstructed EV valuation**: daily EV equals daily market cap plus last-known total debt minus last-known cash.
3. **Static vendor valuation benchmarks**: FMP ratios/metrics forward-filled after the filing lag.
4. **As-of quality controls**: margins, returns, leverage, growth, and balance-sheet quality that should not be daily repriced.

The scoring convention normalizes direction before measuring predictive performance: higher score should imply higher expected forward return.


In [2]:
@dataclass(frozen=True)
class FeatureSpec:
    feature: str
    family: str
    variant: str
    expected_direction: str
    source: str


def _date_indexed(frame: pd.DataFrame) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    out = frame.copy()
    if not isinstance(out.index, pd.DatetimeIndex):
        out.index = pd.to_datetime(out.index, errors="coerce")
    out = out.loc[out.index.notna()].sort_index()
    out.index = pd.DatetimeIndex(out.index).normalize()
    return out


def _slice(frame: pd.DataFrame, start: str | None = None, end: str | None = None) -> pd.DataFrame:
    out = _date_indexed(frame)
    if out.empty:
        return out
    if start:
        out = out.loc[out.index >= pd.Timestamp(start)]
    if end:
        out = out.loc[out.index <= pd.Timestamp(end)]
    return out


def _dedupe_asof(values: pd.Series | pd.DataFrame, *, lag_days: int = FILING_LAG_DAYS):
    out = values.copy()
    out.index = pd.DatetimeIndex(pd.to_datetime(out.index, errors="coerce")).normalize() + pd.Timedelta(days=int(lag_days))
    out = out.loc[out.index.notna()].sort_index()
    return out.loc[~out.index.duplicated(keep="last")]


def _asof_to_daily(values: pd.Series | pd.DataFrame, daily_index: pd.DatetimeIndex, *, lag_days: int = FILING_LAG_DAYS):
    sparse = _dedupe_asof(values, lag_days=lag_days)
    return sparse.reindex(daily_index, method="ffill")


def _numeric(frame: pd.DataFrame, column: str) -> pd.Series:
    if frame.empty or column not in frame.columns:
        return pd.Series(dtype="float64")
    return pd.to_numeric(frame[column], errors="coerce").replace([np.inf, -np.inf], np.nan)


def _safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    out = numerator / denominator.replace(0, np.nan)
    return out.replace([np.inf, -np.inf], np.nan)


def _add_feature(panel: pd.DataFrame, specs: list[FeatureSpec], name: str, values: pd.Series, *, family: str, variant: str, expected_direction: str, source: str) -> None:
    panel[name] = values.replace([np.inf, -np.inf], np.nan)
    specs.append(FeatureSpec(name, family, variant, expected_direction, source))


In [3]:
MCAP_VALUE_COLUMNS = {
    "revenue": ("income", "revenue"),
    "gross_profit": ("income", "gross_profit"),
    "ebitda": ("income", "ebitda"),
    "ebit": ("income", "ebit"),
    "operating_income": ("income", "operating_income"),
    "net_income": ("income", "net_income"),
    "operating_cash_flow": ("cash", "operating_cash_flow"),
    "free_cash_flow": ("cash", "free_cash_flow"),
    "book_equity": ("balance", "total_stockholders_equity"),
    "tangible_book": ("derived", "tangible_book"),
    "cash": ("balance", "cash_and_cash_equivalents"),
    "cash_and_short_term_investments": ("balance", "cash_and_short_term_investments"),
    "total_debt": ("balance", "total_debt"),
    "net_debt": ("balance", "net_debt"),
}

EV_DENOMINATORS = {
    "revenue": ("income", "revenue"),
    "gross_profit": ("income", "gross_profit"),
    "ebitda": ("income", "ebitda"),
    "ebit": ("income", "ebit"),
    "operating_income": ("income", "operating_income"),
    "operating_cash_flow": ("cash", "operating_cash_flow"),
    "free_cash_flow": ("cash", "free_cash_flow"),
}

STATIC_VENDOR_RATIOS = {
    "static_pe": ("ratios", "price_to_earnings", "lower_is_better"),
    "static_ps": ("ratios", "price_to_sales", "lower_is_better"),
    "static_pb": ("ratios", "price_to_book", "lower_is_better"),
    "static_pocf": ("ratios", "price_to_operating_cash_flow", "lower_is_better"),
    "static_pfcf": ("ratios", "price_to_free_cash_flow", "lower_is_better"),
    "static_debt_to_market_cap": ("ratios", "debt_to_market_cap", "lower_is_better"),
    "static_dividend_yield": ("ratios", "dividend_yield", "higher_is_better"),
    "static_ev_to_sales": ("metrics", "ev_to_sales", "lower_is_better"),
    "static_ev_to_ebitda": ("metrics", "ev_to_ebitda", "lower_is_better"),
    "static_ev_to_ocf": ("metrics", "ev_to_operating_cash_flow", "lower_is_better"),
    "static_ev_to_fcf": ("metrics", "ev_to_free_cash_flow", "lower_is_better"),
}

ASOF_CONTROLS = {
    "gross_profit_margin": ("ratios", "gross_profit_margin", "higher_is_better"),
    "ebitda_margin": ("ratios", "ebitda_margin", "higher_is_better"),
    "operating_profit_margin": ("ratios", "operating_profit_margin", "higher_is_better"),
    "net_profit_margin": ("ratios", "net_profit_margin", "higher_is_better"),
    "current_ratio": ("ratios", "current_ratio", "higher_is_better"),
    "quick_ratio": ("ratios", "quick_ratio", "higher_is_better"),
    "debt_to_assets": ("ratios", "debt_to_assets", "lower_is_better"),
    "debt_to_equity": ("ratios", "debt_to_equity", "lower_is_better"),
    "return_on_assets": ("metrics", "return_on_assets", "higher_is_better"),
    "return_on_equity": ("metrics", "return_on_equity", "higher_is_better"),
    "return_on_invested_capital": ("metrics", "return_on_invested_capital", "higher_is_better"),
    "income_quality": ("metrics", "income_quality", "higher_is_better"),
    "net_debt_to_ebitda": ("metrics", "net_debt_to_ebitda", "lower_is_better"),
}


In [4]:
def load_symbol_inputs(symbol: str) -> dict[str, pd.DataFrame]:
    return {
        "prices": _slice(wh.read_prices(symbol, provider=PROVIDER), START_DATE, END_DATE),
        "historical_market_cap": _slice(wh.read_fundamentals(symbol, section="historical_market_cap", provider=PROVIDER), START_DATE, END_DATE),
        "income": _slice(wh.read_fundamentals(symbol, section="income", provider=PROVIDER), None, END_DATE),
        "balance": _slice(wh.read_fundamentals(symbol, section="balance", provider=PROVIDER), None, END_DATE),
        "cash": _slice(wh.read_fundamentals(symbol, section="cash", provider=PROVIDER), None, END_DATE),
        "ratios": _slice(wh.read_fundamentals(symbol, section="ratios", provider=PROVIDER), None, END_DATE),
        "metrics": _slice(wh.read_fundamentals(symbol, section="metrics", provider=PROVIDER), None, END_DATE),
        "income_growth": _slice(wh.read_fundamentals(symbol, section="income_growth", provider=PROVIDER), None, END_DATE),
        "balance_growth": _slice(wh.read_fundamentals(symbol, section="balance_growth", provider=PROVIDER), None, END_DATE),
        "cash_growth": _slice(wh.read_fundamentals(symbol, section="cash_growth", provider=PROVIDER), None, END_DATE),
    }


def _source_series(inputs: dict[str, pd.DataFrame], source: str, column: str) -> pd.Series:
    if source == "derived" and column == "tangible_book":
        balance = inputs["balance"]
        equity = _numeric(balance, "total_stockholders_equity")
        goodwill_intangible = _numeric(balance, "goodwill_and_intangible_assets")
        if goodwill_intangible.empty:
            goodwill_intangible = _numeric(balance, "goodwill").add(_numeric(balance, "intangible_assets"), fill_value=0)
        return equity.subtract(goodwill_intangible.reindex(equity.index), fill_value=0)
    return _numeric(inputs[source], column)


def build_symbol_panel(symbol: str) -> tuple[pd.DataFrame, list[FeatureSpec], dict[str, object]]:
    inputs = load_symbol_inputs(symbol)
    prices = inputs["prices"]
    mcap = inputs["historical_market_cap"]
    if prices.empty or mcap.empty or "close" not in prices.columns or "market_cap" not in mcap.columns:
        return pd.DataFrame(), [], {"symbol": symbol, "status": "missing_prices_or_market_cap"}

    close = pd.to_numeric(prices["close"], errors="coerce")
    daily_mcap = pd.to_numeric(mcap["market_cap"], errors="coerce").reindex(prices.index, method="ffill")
    panel = pd.DataFrame({"date": prices.index, "symbol": symbol, "close": close, "daily_market_cap": daily_mcap}, index=prices.index)
    specs: list[FeatureSpec] = []

    cash = _asof_to_daily(_source_series(inputs, "balance", "cash_and_cash_equivalents"), panel.index)
    total_debt = _asof_to_daily(_source_series(inputs, "balance", "total_debt"), panel.index)
    net_debt = _asof_to_daily(_source_series(inputs, "balance", "net_debt"), panel.index)
    panel["daily_enterprise_value"] = daily_mcap + total_debt - cash
    panel["asof_total_debt"] = total_debt
    panel["asof_cash"] = cash
    panel["asof_net_debt"] = net_debt

    for label, (source, column) in MCAP_VALUE_COLUMNS.items():
        sparse = _source_series(inputs, source, column)
        if sparse.empty:
            continue
        daily_value = _asof_to_daily(sparse, panel.index)
        if daily_value.notna().sum() < MIN_OBS:
            continue
        yield_name = f"daily_{label}_to_mcap"
        multiple_name = f"daily_mcap_to_{label}"
        _add_feature(panel, specs, yield_name, _safe_divide(daily_value, daily_mcap), family=label, variant="daily_market_cap_yield", expected_direction="higher_is_better", source=f"{source}.{column}")
        _add_feature(panel, specs, multiple_name, _safe_divide(daily_mcap, daily_value), family=label, variant="daily_market_cap_multiple", expected_direction="lower_is_better", source=f"{source}.{column}")

    for label, (source, column) in EV_DENOMINATORS.items():
        sparse = _source_series(inputs, source, column)
        if sparse.empty:
            continue
        daily_value = _asof_to_daily(sparse, panel.index)
        if daily_value.notna().sum() < MIN_OBS:
            continue
        multiple_name = f"daily_ev_to_{label}"
        yield_name = f"daily_{label}_to_ev"
        _add_feature(panel, specs, multiple_name, _safe_divide(panel["daily_enterprise_value"], daily_value), family=f"ev_{label}", variant="daily_ev_multiple", expected_direction="lower_is_better", source=f"{source}.{column}")
        _add_feature(panel, specs, yield_name, _safe_divide(daily_value, panel["daily_enterprise_value"]), family=f"ev_{label}", variant="daily_ev_yield", expected_direction="higher_is_better", source=f"{source}.{column}")

    for name, (source, column, direction) in STATIC_VENDOR_RATIOS.items():
        sparse = _numeric(inputs[source], column)
        if sparse.empty:
            continue
        values = _asof_to_daily(sparse, panel.index)
        if values.notna().sum() < MIN_OBS:
            continue
        _add_feature(panel, specs, name, values, family=name.replace("static_", ""), variant="static_vendor_ratio", expected_direction=direction, source=f"{source}.{column}")

    for name, (source, column, direction) in ASOF_CONTROLS.items():
        sparse = _numeric(inputs[source], column)
        if sparse.empty:
            continue
        values = _asof_to_daily(sparse, panel.index)
        if values.notna().sum() < MIN_OBS:
            continue
        _add_feature(panel, specs, f"asof_{name}", values, family=name, variant="asof_quality_control", expected_direction=direction, source=f"{source}.{column}")

    for horizon in HORIZONS:
        panel[f"forward_return_{horizon}d"] = close.shift(-int(horizon)) / close - 1.0

    diagnostics = {
        "symbol": symbol,
        "status": "ok",
        "price_rows": len(prices),
        "market_cap_rows": len(mcap),
        "market_cap_start": str(mcap.index.min().date()),
        "market_cap_end": str(mcap.index.max().date()),
        "income_rows": len(inputs["income"]),
        "balance_rows": len(inputs["balance"]),
        "cash_rows": len(inputs["cash"]),
        "feature_count": len(specs),
    }
    return panel.reset_index(drop=True), specs, diagnostics


panel_build_start = perf_counter()
frames: list[pd.DataFrame] = []
all_specs: list[FeatureSpec] = []
diagnostics: list[dict[str, object]] = []
for symbol in ANALYSIS_SYMBOLS:
    frame, specs, diag = build_symbol_panel(symbol)
    diagnostics.append(diag)
    if not frame.empty:
        frames.append(frame)
        all_specs.extend(specs)

panel = pd.concat(frames, ignore_index=True).sort_values(["date", "symbol"]).reset_index(drop=True)
feature_metadata = pd.DataFrame([spec.__dict__ for spec in all_specs]).drop_duplicates().sort_values(["variant", "family", "feature"]).reset_index(drop=True)
diagnostics_df = pd.DataFrame(diagnostics)
feature_cols = feature_metadata["feature"].tolist()
run_timings["panel_build_seconds"] = perf_counter() - panel_build_start

print({"symbols": sorted(panel["symbol"].unique()), "rows": len(panel), "features": len(feature_cols), "start": str(panel["date"].min().date()), "end": str(panel["date"].max().date()), "panel_build_seconds": round(run_timings["panel_build_seconds"], 2)})
display(diagnostics_df)
display(feature_metadata.groupby(["variant", "expected_direction"]).size().rename("feature_count").reset_index())
display(feature_metadata.head(30))


{'symbols': ['AAPL', 'MSFT', 'NVDA'], 'rows': 1863, 'features': 59, 'start': '2024-01-02', 'end': '2026-06-24', 'panel_build_seconds': 0.17}


,symbol,status,price_rows,market_cap_rows,market_cap_start,market_cap_end,income_rows,balance_rows,cash_rows,feature_count
0,AAPL,ok,621,618,2024-01-02,2026-06-18,162,150,145,59
1,MSFT,ok,621,618,2024-01-02,2026-06-18,162,151,147,59
2,NVDA,ok,621,618,2024-01-02,2026-06-18,109,109,109,59


,variant,expected_direction,feature_count
0,asof_quality_control,higher_is_better,10
1,asof_quality_control,lower_is_better,1
2,daily_ev_multiple,lower_is_better,7
3,daily_ev_yield,higher_is_better,7
4,daily_market_cap_multiple,lower_is_better,14
5,daily_market_cap_yield,higher_is_better,14
6,static_vendor_ratio,higher_is_better,1
7,static_vendor_ratio,lower_is_better,5


,feature,family,variant,expected_direction,source
0,asof_current_ratio,current_ratio,asof_quality_control,higher_is_better,ratios.current_ratio
1,asof_ebitda_margin,ebitda_margin,asof_quality_control,higher_is_better,ratios.ebitda_margin
2,asof_gross_profit_margin,gross_profit_margin,asof_quality_control,higher_is_better,ratios.gross_profit_margin
3,asof_income_quality,income_quality,asof_quality_control,higher_is_better,metrics.income_quality
4,asof_net_debt_to_ebitda,net_debt_to_ebitda,asof_quality_control,lower_is_better,metrics.net_debt_to_ebitda
5,asof_net_profit_margin,net_profit_margin,asof_quality_control,higher_is_better,ratios.net_profit_margin
6,asof_operating_profit_margin,operating_profit_margin,asof_quality_control,higher_is_better,ratios.operating_profit_margin
7,asof_quick_ratio,quick_ratio,asof_quality_control,higher_is_better,ratios.quick_ratio
8,asof_return_on_assets,return_on_assets,asof_quality_control,higher_is_better,metrics.return_on_assets
9,asof_return_on_equity,return_on_equity,asof_quality_control,higher_is_better,metrics.return_on_equity


## Coverage And Sanity Checks

Before measuring signal, verify that the daily market cap source is actually daily and that the reconstructed EV is available after the first lagged balance sheet observation.


In [5]:
coverage_rows = []
for symbol, group in panel.groupby("symbol"):
    group = group.sort_values("date")
    mcap = group.dropna(subset=["daily_market_cap"])
    ev = group.dropna(subset=["daily_enterprise_value"])
    gaps = pd.to_datetime(mcap["date"]).diff().dt.days.dropna()
    coverage_rows.append({
        "symbol": symbol,
        "panel_rows": len(group),
        "mcap_non_null": int(group["daily_market_cap"].notna().sum()),
        "ev_non_null": int(group["daily_enterprise_value"].notna().sum()),
        "mcap_start": mcap["date"].min(),
        "mcap_end": mcap["date"].max(),
        "mcap_median_gap_days": float(gaps.median()) if not gaps.empty else np.nan,
        "mcap_mode_gap_days": int(gaps.mode().iloc[0]) if not gaps.empty else np.nan,
        "latest_market_cap": mcap["daily_market_cap"].iloc[-1] if not mcap.empty else np.nan,
        "latest_daily_ev": ev["daily_enterprise_value"].iloc[-1] if not ev.empty else np.nan,
    })
coverage = pd.DataFrame(coverage_rows).sort_values("symbol")
display(coverage)

sample_cols = ["symbol", "date", "close", "daily_market_cap", "daily_enterprise_value", "asof_total_debt", "asof_cash"]
display(panel[sample_cols].dropna().groupby("symbol").tail(2))


,symbol,panel_rows,mcap_non_null,ev_non_null,mcap_start,mcap_end,mcap_median_gap_days,mcap_mode_gap_days,latest_market_cap,latest_daily_ev
0,AAPL,621,621,621,2024-01-02,2026-06-24,1.0000,1,4383941071180,4432324071180
1,MSFT,621,621,621,2024-01-02,2026-06-24,1.0000,1,2817424400000,2842284400000
2,NVDA,621,621,621,2024-01-02,2026-06-24,1.0000,1,5116817340000,5117624340000


,symbol,date,close,daily_market_cap,daily_enterprise_value,asof_total_debt,asof_cash
1857,AAPL,2026-06-23,294.3000,4383941071180,4432324071180,84711000000,36328000000
1858,MSFT,2026-06-23,373.9400,2817424400000,2842284400000,56965000000,32105000000
1859,NVDA,2026-06-23,200.0400,5116817340000,5117624340000,11412000000,10605000000
1860,AAPL,2026-06-24,293.0800,4383941071180,4432324071180,84711000000,36328000000
1861,MSFT,2026-06-24,365.4600,2817424400000,2842284400000,56965000000,32105000000
1862,NVDA,2026-06-24,199.0000,5116817340000,5117624340000,11412000000,10605000000


## Correctness And Runtime Metrics

For the smoke run, the actual small-sample rank IC values are not the point. This section checks that:

- The full feature panel can be evaluated end to end.
- A simple baseline evaluator and the optimized evaluator produce matching rank IC values on a subset.
- The optimized evaluator is fast enough to justify scaling to more symbols or longer history.

The baseline subset intentionally uses only a few features to keep the notebook quick. The optimized evaluator runs all engineered features.

In [6]:
def _spearman(a: pd.Series, b: pd.Series) -> float:
    valid = pd.concat([a, b], axis=1).dropna()
    if len(valid) < 3 or valid.iloc[:, 0].nunique() < 2 or valid.iloc[:, 1].nunique() < 2:
        return np.nan
    return float(valid.iloc[:, 0].rank().corr(valid.iloc[:, 1].rank()))


def _score_direction(values: pd.Series | pd.DataFrame, expected_direction: str):
    if expected_direction == "lower_is_better":
        return -values
    return values


def _prepare_score_panel(features: list[str]) -> pd.DataFrame:
    direction_sign = feature_metadata.set_index("feature").loc[features, "expected_direction"].map({"higher_is_better": 1.0, "lower_is_better": -1.0})
    out = panel[["date", "symbol", *features]].copy()
    out.loc[:, features] = out[features].multiply(direction_sign, axis="columns")
    return out


def evaluate_baseline_loop(features: list[str], horizons: tuple[int, ...]) -> pd.DataFrame:
    rows = []
    score_frame = _prepare_score_panel(features)
    for horizon in horizons:
        return_col = f"forward_return_{horizon}d"
        eval_frame = score_frame[["date", "symbol", *features]].copy()
        eval_frame[return_col] = panel[return_col]
        for feature in features:
            daily_ic = []
            daily_spread = []
            for _, group in eval_frame[["date", "symbol", feature, return_col]].groupby("date", sort=False):
                valid = group[[feature, return_col]].replace([np.inf, -np.inf], np.nan).dropna()
                if len(valid) < 3 or valid[feature].nunique() < 2 or valid[return_col].nunique() < 2:
                    continue
                daily_ic.append(_spearman(valid[feature], valid[return_col]))
                n = 1 if len(valid) < 4 else 2
                ranked = valid.sort_values(feature)
                daily_spread.append(ranked.tail(n)[return_col].mean() - ranked.head(n)[return_col].mean())
            daily_ic_s = pd.Series(daily_ic, dtype="float64").dropna()
            daily_spread_s = pd.Series(daily_spread, dtype="float64").dropna()
            valid_all = eval_frame[[feature, return_col]].replace([np.inf, -np.inf], np.nan).dropna()
            rows.append({
                "feature": feature,
                "horizon": horizon,
                "observations": int(len(valid_all)),
                "days": int(len(daily_ic_s)),
                "pooled_spearman": _spearman(valid_all[feature], valid_all[return_col]),
                "mean_daily_rank_ic": float(daily_ic_s.mean()) if not daily_ic_s.empty else np.nan,
                "median_daily_rank_ic": float(daily_ic_s.median()) if not daily_ic_s.empty else np.nan,
                "rank_ic_hit_rate": float((daily_ic_s > 0).mean()) if not daily_ic_s.empty else np.nan,
                "mean_top_minus_bottom": float(daily_spread_s.mean()) if not daily_spread_s.empty else np.nan,
                "median_top_minus_bottom": float(daily_spread_s.median()) if not daily_spread_s.empty else np.nan,
            })
    return pd.DataFrame(rows)


def evaluate_vectorized(features: list[str], horizons: tuple[int, ...]) -> pd.DataFrame:
    score_frame = _prepare_score_panel(features)
    rows = []
    for horizon in horizons:
        return_col = f"forward_return_{horizon}d"
        eval_frame = score_frame[["date", "symbol", *features]].copy()
        eval_frame[return_col] = panel[return_col]
        daily_ic_rows = []
        daily_spread_rows = []

        for date, group in eval_frame.groupby("date", sort=False):
            returns = pd.to_numeric(group[return_col], errors="coerce")
            if returns.notna().sum() < 3 or returns.nunique(dropna=True) < 2:
                continue
            scores = group[features].replace([np.inf, -np.inf], np.nan)
            valid_counts = scores.notna().sum(axis=0)
            score_ranks = scores.rank(axis=0)
            return_rank = returns.rank()
            daily_ic = score_ranks.corrwith(return_rank)
            daily_ic.name = date
            daily_ic_rows.append(daily_ic)

            n_by_feature = valid_counts.apply(lambda count: 1 if count < 4 else 2)
            top_mask = score_ranks.gt(valid_counts - n_by_feature, axis="columns")
            bottom_mask = score_ranks.le(n_by_feature, axis="columns")
            returns_matrix = pd.DataFrame(np.tile(returns.to_numpy().reshape(-1, 1), (1, len(features))), index=scores.index, columns=features)
            daily_spread = returns_matrix.where(top_mask).mean(axis=0) - returns_matrix.where(bottom_mask).mean(axis=0)
            daily_spread.loc[valid_counts < 3] = np.nan
            daily_spread.name = date
            daily_spread_rows.append(daily_spread)

        daily_ic_df = pd.DataFrame(daily_ic_rows)
        daily_spread_df = pd.DataFrame(daily_spread_rows)
        for feature in features:
            valid = eval_frame[[feature, return_col]].replace([np.inf, -np.inf], np.nan).dropna()
            daily_ic = daily_ic_df[feature].dropna() if feature in daily_ic_df else pd.Series(dtype="float64")
            daily_spread = daily_spread_df[feature].dropna() if feature in daily_spread_df else pd.Series(dtype="float64")
            rows.append({
                "feature": feature,
                "horizon": horizon,
                "observations": int(len(valid)),
                "days": int(daily_ic.size),
                "pooled_spearman": _spearman(valid[feature], valid[return_col]),
                "mean_daily_rank_ic": float(daily_ic.mean()) if not daily_ic.empty else np.nan,
                "median_daily_rank_ic": float(daily_ic.median()) if not daily_ic.empty else np.nan,
                "rank_ic_hit_rate": float((daily_ic > 0).mean()) if not daily_ic.empty else np.nan,
                "mean_top_minus_bottom": float(daily_spread.mean()) if not daily_spread.empty else np.nan,
                "median_top_minus_bottom": float(daily_spread.median()) if not daily_spread.empty else np.nan,
            })
    return pd.DataFrame(rows)


benchmark_features = feature_cols[: min(12, len(feature_cols))]
baseline_start = perf_counter()
baseline_subset_results = evaluate_baseline_loop(benchmark_features, HORIZONS)
run_timings["baseline_subset_seconds"] = perf_counter() - baseline_start

optimized_subset_start = perf_counter()
optimized_subset_results = evaluate_vectorized(benchmark_features, HORIZONS)
run_timings["optimized_subset_seconds"] = perf_counter() - optimized_subset_start

optimized_full_start = perf_counter()
results_df = evaluate_vectorized(feature_cols, HORIZONS)
run_timings["evaluation_seconds"] = perf_counter() - optimized_full_start

results_df = results_df.merge(feature_metadata, on="feature", how="left")
results_df["abs_mean_daily_rank_ic"] = results_df["mean_daily_rank_ic"].abs()
results_df["spread_bps"] = results_df["mean_top_minus_bottom"] * 10000
results_df = results_df.dropna(subset=["mean_daily_rank_ic"]).reset_index(drop=True)

benchmark_compare = baseline_subset_results.merge(
    optimized_subset_results,
    on=["feature", "horizon"],
    suffixes=("_baseline", "_optimized"),
)
max_metric_diff = (
    benchmark_compare["mean_daily_rank_ic_baseline"] - benchmark_compare["mean_daily_rank_ic_optimized"]
).abs().max()

runtime_benchmark = pd.DataFrame([
    {
        "method": "baseline_loop_subset",
        "features": len(benchmark_features),
        "horizons": len(HORIZONS),
        "seconds": run_timings["baseline_subset_seconds"],
        "seconds_per_feature_horizon": run_timings["baseline_subset_seconds"] / max(len(benchmark_features) * len(HORIZONS), 1),
    },
    {
        "method": "optimized_vectorized_subset",
        "features": len(benchmark_features),
        "horizons": len(HORIZONS),
        "seconds": run_timings["optimized_subset_seconds"],
        "seconds_per_feature_horizon": run_timings["optimized_subset_seconds"] / max(len(benchmark_features) * len(HORIZONS), 1),
    },
    {
        "method": "optimized_vectorized_full",
        "features": len(feature_cols),
        "horizons": len(HORIZONS),
        "seconds": run_timings["evaluation_seconds"],
        "seconds_per_feature_horizon": run_timings["evaluation_seconds"] / max(len(feature_cols) * len(HORIZONS), 1),
    },
])

summary_by_variant = (
    results_df.groupby(["horizon", "variant"])
    .agg(
        features=("feature", "nunique"),
        median_rank_ic=("mean_daily_rank_ic", "median"),
        mean_rank_ic=("mean_daily_rank_ic", "mean"),
        median_spread_bps=("spread_bps", "median"),
        positive_ic_share=("mean_daily_rank_ic", lambda x: float((x > 0).mean())),
    )
    .reset_index()
    .sort_values(["horizon", "median_rank_ic"], ascending=[True, False])
)

top_features = (
    results_df.sort_values(["horizon", "mean_daily_rank_ic", "mean_top_minus_bottom"], ascending=[True, False, False])
    .groupby("horizon")
    .head(15)
    [["horizon", "feature", "variant", "family", "expected_direction", "mean_daily_rank_ic", "median_daily_rank_ic", "rank_ic_hit_rate", "spread_bps", "observations"]]
)

worst_features = (
    results_df.sort_values(["horizon", "mean_daily_rank_ic", "mean_top_minus_bottom"], ascending=[True, True, True])
    .groupby("horizon")
    .head(10)
    [["horizon", "feature", "variant", "family", "expected_direction", "mean_daily_rank_ic", "median_daily_rank_ic", "rank_ic_hit_rate", "spread_bps", "observations"]]
)

print({
    "optimized_full_seconds": round(run_timings["evaluation_seconds"], 3),
    "baseline_subset_seconds": round(run_timings["baseline_subset_seconds"], 3),
    "optimized_subset_seconds": round(run_timings["optimized_subset_seconds"], 3),
    "max_rank_ic_diff_baseline_vs_optimized": None if pd.isna(max_metric_diff) else round(float(max_metric_diff), 12),
    "feature_horizon_results": len(results_df),
})
display(runtime_benchmark)
display(summary_by_variant)
display(top_features)
display(worst_features)


{'optimized_full_seconds': 3.872, 'baseline_subset_seconds': 12.611, 'optimized_subset_seconds': 1.75, 'max_rank_ic_diff_baseline_vs_optimized': 0.0, 'feature_horizon_results': 177}


,method,features,horizons,seconds,seconds_per_feature_horizon
0,baseline_loop_subset,12,3,12.6111,0.3503
1,optimized_vectorized_subset,12,3,1.7500,0.0486
2,optimized_vectorized_full,59,3,3.8723,0.0219


,horizon,variant,features,median_rank_ic,mean_rank_ic,median_spread_bps,positive_ic_share
0,20,asof_quality_control,11,0.0740,0.0803,338.3901,0.9091
5,20,static_vendor_ratio,6,-0.1485,-0.1138,-398.7100,0.3333
1,20,daily_ev_multiple,7,-0.1647,-0.1426,-460.7226,0.0000
2,20,daily_ev_yield,7,-0.1647,-0.1426,-460.7226,0.0000
3,20,daily_market_cap_multiple,14,-0.1676,-0.1430,-443.6628,0.0000
4,20,daily_market_cap_yield,14,-0.1689,-0.1481,-445.1963,0.0000
6,60,asof_quality_control,11,0.1337,0.1639,822.3737,0.9091
11,60,static_vendor_ratio,6,-0.2634,-0.2014,-991.7897,0.3333
7,60,daily_ev_multiple,7,-0.2968,-0.2813,"-1,015.9784",0.0000
8,60,daily_ev_yield,7,-0.2968,-0.2813,"-1,015.9784",0.0000


,horizon,feature,variant,family,expected_direction,mean_daily_rank_ic,median_daily_rank_ic,rank_ic_hit_rate,spread_bps,observations
10,20,asof_return_on_invested_capital,asof_quality_control,return_on_invested_capital,higher_is_better,0.2230,0.5000,0.6672,488.0937,1803
8,20,asof_return_on_assets,asof_quality_control,return_on_assets,higher_is_better,0.1897,0.5000,0.6439,458.8150,1803
9,20,asof_return_on_equity,asof_quality_control,return_on_equity,higher_is_better,0.1148,0.5000,0.5591,101.9876,1803
4,20,asof_net_debt_to_ebitda,asof_quality_control,net_debt_to_ebitda,lower_is_better,0.1040,0.5000,0.5424,342.4679,1803
0,20,asof_current_ratio,asof_quality_control,current_ratio,higher_is_better,0.0740,0.5000,0.5308,338.3901,1803
5,20,asof_net_profit_margin,asof_quality_control,net_profit_margin,higher_is_better,0.0740,0.5000,0.5308,338.3901,1803
6,20,asof_operating_profit_margin,asof_quality_control,operating_profit_margin,higher_is_better,0.0740,0.5000,0.5308,338.3901,1803
7,20,asof_quick_ratio,asof_quality_control,quick_ratio,higher_is_better,0.0740,0.5000,0.5308,338.3901,1803
53,20,static_debt_to_market_cap,static_vendor_ratio,debt_to_market_cap,lower_is_better,0.0740,0.5000,0.5308,338.3901,1803
1,20,asof_ebitda_margin,asof_quality_control,ebitda_margin,higher_is_better,0.0349,0.5000,0.5125,290.9248,1803


,horizon,feature,variant,family,expected_direction,mean_daily_rank_ic,median_daily_rank_ic,rank_ic_hit_rate,spread_bps,observations
29,20,daily_mcap_to_ebitda,daily_market_cap_multiple,ebitda,lower_is_better,-0.2404,-0.5000,0.3444,-539.2509,1803
43,20,daily_ebitda_to_mcap,daily_market_cap_yield,ebitda,higher_is_better,-0.2404,-0.5000,0.3444,-539.2509,1803
57,20,static_ev_to_ocf,static_vendor_ratio,ev_to_ocf,lower_is_better,-0.2379,-0.5000,0.3428,-459.0298,1803
55,20,static_ev_to_ebitda,static_vendor_ratio,ev_to_ebitda,lower_is_better,-0.2313,-0.5000,0.3161,-524.3575,1803
12,20,daily_ev_to_ebitda,daily_ev_multiple,ev_ebitda,lower_is_better,-0.2271,-0.5000,0.3527,-524.4058,1803
19,20,daily_ebitda_to_ev,daily_ev_yield,ev_ebitda,higher_is_better,-0.2271,-0.5000,0.3527,-524.4058,1803
54,20,static_dividend_yield,static_vendor_ratio,dividend_yield,higher_is_better,-0.2230,-0.5000,0.3328,-488.0937,1803
46,20,daily_net_debt_to_mcap,daily_market_cap_yield,net_debt,higher_is_better,-0.2038,-0.5000,0.3794,-478.7995,1803
34,20,daily_mcap_to_operating_cash_flow,daily_market_cap_multiple,operating_cash_flow,lower_is_better,-0.1922,-0.5000,0.3644,-486.2007,1803
48,20,daily_operating_cash_flow_to_mcap,daily_market_cap_yield,operating_cash_flow,higher_is_better,-0.1922,-0.5000,0.3644,-486.2007,1803


## Static Vendor Ratios Vs Daily-Adjusted Rebuilds

This table compares similar valuation families where FMP provides a static/as-of ratio and the notebook reconstructs a daily-adjusted equivalent from daily market cap or daily EV.


In [7]:
comparison_map = {
    "pe": ["static_pe", "daily_net_income_to_mcap", "daily_mcap_to_net_income"],
    "ps": ["static_ps", "daily_revenue_to_mcap", "daily_mcap_to_revenue"],
    "pb": ["static_pb", "daily_book_equity_to_mcap", "daily_mcap_to_book_equity"],
    "pocf": ["static_pocf", "daily_operating_cash_flow_to_mcap", "daily_mcap_to_operating_cash_flow"],
    "pfcf": ["static_pfcf", "daily_free_cash_flow_to_mcap", "daily_mcap_to_free_cash_flow"],
    "ev_sales": ["static_ev_to_sales", "daily_ev_to_revenue", "daily_revenue_to_ev"],
    "ev_ebitda": ["static_ev_to_ebitda", "daily_ev_to_ebitda", "daily_ebitda_to_ev"],
    "ev_ocf": ["static_ev_to_ocf", "daily_ev_to_operating_cash_flow", "daily_operating_cash_flow_to_ev"],
    "ev_fcf": ["static_ev_to_fcf", "daily_ev_to_free_cash_flow", "daily_free_cash_flow_to_ev"],
}

comparison_rows = []
for family, features in comparison_map.items():
    for feature in features:
        subset = results_df.loc[results_df["feature"].eq(feature)]
        if subset.empty:
            continue
        for _, row in subset.iterrows():
            comparison_rows.append({
                "comparison_family": family,
                "horizon": row["horizon"],
                "feature": feature,
                "variant": row["variant"],
                "mean_daily_rank_ic": row["mean_daily_rank_ic"],
                "rank_ic_hit_rate": row["rank_ic_hit_rate"],
                "spread_bps": row["spread_bps"],
            })
comparison_df = pd.DataFrame(comparison_rows).sort_values(["comparison_family", "horizon", "mean_daily_rank_ic"], ascending=[True, True, False])
display(comparison_df)

winner_df = (
    comparison_df.sort_values(["comparison_family", "horizon", "mean_daily_rank_ic"], ascending=[True, True, False])
    .groupby(["comparison_family", "horizon"])
    .head(1)
    .reset_index(drop=True)
)
display(winner_df)


,comparison_family,horizon,feature,variant,mean_daily_rank_ic,rank_ic_hit_rate,spread_bps
42,ev_ebitda,20,daily_ev_to_ebitda,daily_ev_multiple,-0.2271,0.3527,-524.4058
45,ev_ebitda,20,daily_ebitda_to_ev,daily_ev_yield,-0.2271,0.3527,-524.4058
39,ev_ebitda,20,static_ev_to_ebitda,static_vendor_ratio,-0.2313,0.3161,-524.3575
43,ev_ebitda,60,daily_ev_to_ebitda,daily_ev_multiple,-0.4109,0.2193,"-1,232.8944"
46,ev_ebitda,60,daily_ebitda_to_ev,daily_ev_yield,-0.4109,0.2193,"-1,232.8944"
...,...,...,...,...,...,...,...
9,ps,20,daily_mcap_to_revenue,daily_market_cap_multiple,-0.0674,0.4725,-332.2349
7,ps,60,daily_revenue_to_mcap,daily_market_cap_yield,-0.1390,0.4064,-829.4538
10,ps,60,daily_mcap_to_revenue,daily_market_cap_multiple,-0.1390,0.4064,-829.4538
8,ps,120,daily_revenue_to_mcap,daily_market_cap_yield,-0.2116,0.3713,"-1,471.4333"


,comparison_family,horizon,feature,variant,mean_daily_rank_ic,rank_ic_hit_rate,spread_bps
0,ev_ebitda,20,daily_ev_to_ebitda,daily_ev_multiple,-0.2271,0.3527,-524.4058
1,ev_ebitda,60,daily_ev_to_ebitda,daily_ev_multiple,-0.4109,0.2193,"-1,232.8944"
2,ev_ebitda,120,daily_ev_to_ebitda,daily_ev_multiple,-0.4052,0.2036,"-1,938.1187"
3,ev_fcf,20,static_ev_to_fcf,static_vendor_ratio,0.0092,0.4925,-112.0940
4,ev_fcf,60,static_ev_to_fcf,static_vendor_ratio,0.0731,0.5455,-174.6414
5,ev_fcf,120,static_ev_to_fcf,static_vendor_ratio,-0.0269,0.4790,-600.1409
6,ev_ocf,20,daily_ev_to_operating_cash_flow,daily_ev_multiple,-0.1897,0.3644,-484.9102
7,ev_ocf,60,static_ev_to_ocf,static_vendor_ratio,-0.3930,0.2852,"-1,161.2057"
8,ev_ocf,120,daily_ev_to_operating_cash_flow,daily_ev_multiple,-0.4511,0.1876,"-2,097.8792"
9,ev_sales,20,daily_ev_to_revenue,daily_ev_multiple,-0.0682,0.4725,-332.8990


## Feature Robustness Across Horizons

A feature is more interesting when it stays useful across multiple horizons, not just one window. This table ranks features by average rank IC across 20, 60, and 120 trading days.


In [8]:
robustness = (
    results_df.groupby(["feature", "variant", "family", "expected_direction"])
    .agg(
        horizons=("horizon", "nunique"),
        avg_rank_ic=("mean_daily_rank_ic", "mean"),
        min_rank_ic=("mean_daily_rank_ic", "min"),
        avg_spread_bps=("spread_bps", "mean"),
        positive_horizons=("mean_daily_rank_ic", lambda x: int((x > 0).sum())),
    )
    .reset_index()
    .sort_values(["positive_horizons", "avg_rank_ic", "avg_spread_bps"], ascending=[False, False, False])
)
display(robustness.head(25))
display(robustness.tail(15))


,feature,variant,family,expected_direction,horizons,avg_rank_ic,min_rank_ic,avg_spread_bps,positive_horizons
8,asof_return_on_assets,asof_quality_control,return_on_assets,higher_is_better,3,0.4322,0.1897,"1,528.9742",3
10,asof_return_on_invested_capital,asof_quality_control,return_on_invested_capital,higher_is_better,3,0.4113,0.2230,"1,434.5212",3
9,asof_return_on_equity,asof_quality_control,return_on_equity,higher_is_better,3,0.2705,0.1148,614.6036,3
4,asof_net_debt_to_ebitda,asof_quality_control,net_debt_to_ebitda,lower_is_better,3,0.2147,0.1040,"1,002.1142",3
0,asof_current_ratio,asof_quality_control,current_ratio,higher_is_better,3,0.1398,0.0740,877.3990,3
5,asof_net_profit_margin,asof_quality_control,net_profit_margin,higher_is_better,3,0.1398,0.0740,877.3990,3
6,asof_operating_profit_margin,asof_quality_control,operating_profit_margin,higher_is_better,3,0.1398,0.0740,877.3990,3
7,asof_quick_ratio,asof_quality_control,quick_ratio,higher_is_better,3,0.1398,0.0740,877.3990,3
53,static_debt_to_market_cap,static_vendor_ratio,debt_to_market_cap,lower_is_better,3,0.1398,0.0740,877.3990,3
2,asof_gross_profit_margin,asof_quality_control,gross_profit_margin,higher_is_better,3,0.0856,0.0341,739.2803,3


,feature,variant,family,expected_direction,horizons,avg_rank_ic,min_rank_ic,avg_spread_bps,positive_horizons
3,asof_income_quality,asof_quality_control,income_quality,higher_is_better,3,-0.2879,-0.4501,-987.3317,0
16,daily_ebitda_to_ev,daily_ev_yield,ev_ebitda,higher_is_better,3,-0.3477,-0.4109,"-1,231.8063",0
19,daily_ev_to_ebitda,daily_ev_multiple,ev_ebitda,lower_is_better,3,-0.3477,-0.4109,"-1,231.8063",0
38,daily_mcap_to_operating_cash_flow,daily_market_cap_multiple,operating_cash_flow,lower_is_better,3,-0.3513,-0.4481,"-1,278.6199",0
46,daily_operating_cash_flow_to_mcap,daily_market_cap_yield,operating_cash_flow,higher_is_better,3,-0.3513,-0.4481,"-1,278.6199",0
22,daily_ev_to_operating_cash_flow,daily_ev_multiple,ev_operating_cash_flow,lower_is_better,3,-0.3523,-0.4511,"-1,281.0401",0
45,daily_operating_cash_flow_to_ev,daily_ev_yield,ev_operating_cash_flow,higher_is_better,3,-0.3523,-0.4511,"-1,281.0401",0
43,daily_net_debt_to_mcap,daily_market_cap_yield,net_debt,higher_is_better,3,-0.3537,-0.4321,"-1,291.0351",0
17,daily_ebitda_to_mcap,daily_market_cap_yield,ebitda,higher_is_better,3,-0.3551,-0.4207,"-1,239.3378",0
33,daily_mcap_to_ebitda,daily_market_cap_multiple,ebitda,lower_is_better,3,-0.3551,-0.4207,"-1,239.3378",0


## Runtime Scaling Estimate

The smoke test runs first. This section estimates how runtime could scale when expanding to all MAG7 symbols and a longer history. It is a projection from observed panel size, not a guarantee; if the projected runtime gets too high, optimize before scaling.

In [9]:
smoke_rows = len(panel)
smoke_symbols = len(ANALYSIS_SYMBOLS)
smoke_features = len(feature_cols)
smoke_start = pd.Timestamp(START_DATE)
smoke_end = pd.Timestamp(panel["date"].max())
smoke_days = max((smoke_end - smoke_start).days, 1)

full_start = pd.Timestamp(FULL_RUN_BASELINE_START_DATE)
full_days = max((smoke_end - full_start).days, 1)
full_symbol_factor = len(FULL_RUN_BASELINE_SYMBOLS) / max(smoke_symbols, 1)
full_history_factor = full_days / smoke_days
projected_full_rows = int(smoke_rows * full_symbol_factor * full_history_factor)

observed_total_seconds = run_timings.get("panel_build_seconds", 0.0) + run_timings.get("evaluation_seconds", 0.0)
projected_full_seconds = observed_total_seconds * full_symbol_factor * full_history_factor
projection = pd.DataFrame([
    {
        "run": "smoke_current",
        "symbols": smoke_symbols,
        "start_date": START_DATE,
        "rows": smoke_rows,
        "features": smoke_features,
        "panel_build_seconds": run_timings.get("panel_build_seconds", np.nan),
        "evaluation_seconds": run_timings.get("evaluation_seconds", np.nan),
        "total_seconds": observed_total_seconds,
    },
    {
        "run": "projected_mag7_since_2018",
        "symbols": len(FULL_RUN_BASELINE_SYMBOLS),
        "start_date": FULL_RUN_BASELINE_START_DATE,
        "rows": projected_full_rows,
        "features": smoke_features,
        "panel_build_seconds": run_timings.get("panel_build_seconds", np.nan) * full_symbol_factor * full_history_factor,
        "evaluation_seconds": run_timings.get("evaluation_seconds", np.nan) * full_symbol_factor * full_history_factor,
        "total_seconds": projected_full_seconds,
    },
])
projection["total_minutes"] = projection["total_seconds"] / 60.0
display(projection)

if projected_full_seconds > 120:
    display(Markdown("> Projected full run is above two minutes. Optimize before scaling: cache warehouse reads, reduce displayed intermediate tables, and vectorize top/bottom spread selection."))
else:
    display(Markdown("> Projected full MAG7 run is acceptable for a notebook. Larger universes should still move feature construction/evaluation into reusable Python helpers."))


,run,symbols,start_date,rows,features,panel_build_seconds,evaluation_seconds,total_seconds,total_minutes
0,smoke_current,3,2024-01-01,1863,59,0.1697,3.8723,4.0421,0.0674
1,projected_mag7_since_2018,7,2018-01-01,14871,59,1.3549,30.9102,32.2651,0.5378


> Projected full MAG7 run is acceptable for a notebook. Larger universes should still move feature construction/evaluation into reusable Python helpers.

## Written Analysis

This cell is filled after the notebook is executed, using the computed tables above.


In [10]:
def _fmt_seconds(value: float) -> str:
    if pd.isna(value):
        return "nan"
    return f"{value:,.2f}s"

smoke_total = run_timings.get("panel_build_seconds", 0.0) + run_timings.get("evaluation_seconds", 0.0)
projection_row = projection.loc[projection["run"].eq("projected_mag7_since_2018")].iloc[0]
optimized_subset = runtime_benchmark.loc[runtime_benchmark["method"].eq("optimized_vectorized_subset")].iloc[0]
baseline_subset = runtime_benchmark.loc[runtime_benchmark["method"].eq("baseline_loop_subset")].iloc[0]
full_optimized = runtime_benchmark.loc[runtime_benchmark["method"].eq("optimized_vectorized_full")].iloc[0]

analysis_md = "\n".join([
    "### Smoke-Test Analysis From This Run",
    "",
    f"The notebook completed end to end on {panel['symbol'].nunique()} symbols, {len(panel):,} symbol-days, {len(feature_cols)} engineered features, and {len(HORIZONS)} forward-return horizons.",
    f"Panel construction took {_fmt_seconds(run_timings.get('panel_build_seconds', np.nan))}; optimized full evaluation took {_fmt_seconds(run_timings.get('evaluation_seconds', np.nan))}; total measured core runtime was {_fmt_seconds(smoke_total)}.",
    "",
    "#### Correctness Checks",
    f"- Daily historical market cap is present for every smoke symbol and behaves like trading-day data.",
    f"- Daily enterprise value was reconstructed from historical market cap plus lagged debt minus lagged cash; no `historical_enterprise_value`, snapshot share-statistics, or direct FMP API calls were used.",
    f"- Baseline-loop and optimized-vectorized evaluators matched on the benchmark subset; max mean-rank-IC difference was {max_metric_diff:.12f}.",
    "",
    "#### Runtime Checks",
    f"- Baseline subset: {int(baseline_subset['features'])} features x {len(HORIZONS)} horizons in {_fmt_seconds(baseline_subset['seconds'])}.",
    f"- Optimized subset: same subset in {_fmt_seconds(optimized_subset['seconds'])}.",
    f"- Optimized full smoke run: {int(full_optimized['features'])} features x {len(HORIZONS)} horizons in {_fmt_seconds(full_optimized['seconds'])}.",
    f"- Projected MAG7 since 2018 runtime from this smoke profile: {_fmt_seconds(float(projection_row['total_seconds']))} ({projection_row['total_minutes']:.2f} minutes).",
    "",
    "#### Scaling Decision",
    "This smoke run is now useful as an engineering harness. If the projection stays below a few minutes, running MAG7 full history in a notebook is acceptable. If the universe grows beyond MAG7, move the evaluator into a Python module, cache the daily feature panel, and avoid rendering large intermediate tables in the notebook.",
    "",
    "#### Research Caution",
    "Do not interpret the three-symbol rank IC values as trading evidence. The small sample only proves that the feature construction and evaluation path works and gives a first runtime estimate. The real signal test should use a larger liquid universe with sector-neutral ranking.",
])

display(Markdown(analysis_md))


### Smoke-Test Analysis From This Run

The notebook completed end to end on 3 symbols, 1,863 symbol-days, 59 engineered features, and 3 forward-return horizons.
Panel construction took 0.17s; optimized full evaluation took 3.87s; total measured core runtime was 4.04s.

#### Correctness Checks
- Daily historical market cap is present for every smoke symbol and behaves like trading-day data.
- Daily enterprise value was reconstructed from historical market cap plus lagged debt minus lagged cash; no `historical_enterprise_value`, snapshot share-statistics, or direct FMP API calls were used.
- Baseline-loop and optimized-vectorized evaluators matched on the benchmark subset; max mean-rank-IC difference was 0.000000000000.

#### Runtime Checks
- Baseline subset: 12 features x 3 horizons in 12.61s.
- Optimized subset: same subset in 1.75s.
- Optimized full smoke run: 59 features x 3 horizons in 3.87s.
- Projected MAG7 since 2018 runtime from this smoke profile: 32.27s (0.54 minutes).

#### Scaling Decision
This smoke run is now useful as an engineering harness. If the projection stays below a few minutes, running MAG7 full history in a notebook is acceptable. If the universe grows beyond MAG7, move the evaluator into a Python module, cache the daily feature panel, and avoid rendering large intermediate tables in the notebook.

#### Research Caution
Do not interpret the three-symbol rank IC values as trading evidence. The small sample only proves that the feature construction and evaluation path works and gives a first runtime estimate. The real signal test should use a larger liquid universe with sector-neutral ranking.